# Forward SDE Funnel

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

Two data clusters spread into noise under a forward OU SDE.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/forward-sde/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

torch.manual_seed(42)

# The demo's defaults: dX = -theta X dt + sigma dW in 2D, start in two clusters at (-2, 0) and (2, 0)
T, dt, theta, sigma = 4.0, 0.02, 0.8, 0.9
n = 20000
side = torch.randint(0, 2, (n,)) * 2 - 1
x = torch.stack([2.0 * side + 0.35 * torch.randn(n), 0.35 * torch.randn(n)], 1).double()
stat_var = sigma ** 2 / (2 * theta)

print("   t    mean x for right cluster   theory 2 e^{-theta t}   var(x)    fraction x > 0 still on its side")
for k in range(round(T / dt) + 1):
    t = k * dt
    if k in (0, 25, 50, 100, 200):
        right = side == 1
        same = ((x[:, 0] > 0) == right).double().mean().item()
        print(f"{t:5.2f}        {x[right, 0].mean().item():+.4f}              {2 * math.exp(-theta * t):+.4f}         "
              f"{x[:, 0].var().item():.4f}     {same:.3f}")
    x = x - theta * x * dt + sigma * math.sqrt(dt) * torch.randn_like(x)

print(f"\nstationary law N(0, sigma^2/(2 theta) I) = N(0, {stat_var:.4f} I); final covariance",
      [[round(v, 3) for v in r] for r in torch.cov(x.T).tolist()])
# This is the forward (noising) process of a diffusion model: structure is forgotten
# at rate theta, and every start ends at the same Gaussian.

   t    mean x for right cluster   theory 2 e^{-theta t}   var(x)    fraction x > 0 still on its side
 0.00        +1.9974              +2.0000         4.1264     1.000
 0.50        +1.3291              +1.3406         2.1144     0.989
 1.00        +0.8901              +0.8987         1.2182     0.913
 2.00        +0.3958              +0.4038         0.6484     0.716
 4.00        +0.0700              +0.0815         0.5136     0.543

stationary law N(0, sigma^2/(2 theta) I) = N(0, 0.5062 I); final covariance [[0.513, 0.001], [0.001, 0.513]]
